In [1]:
# CELL 1: INSTALL DEPENDENCIES

%pip install -q wordcloud emoji
%pip install -q -U transformers accelerate
%pip install -q imbalanced-learn

In [2]:
# CELL 2: MOUNT DRIVE & PATHS

from google.colab import drive
drive.mount('/content/drive')

GLOVE_PATH = "/content/drive/MyDrive/glove.twitter.27B.100d.txt"
CSV_PATH   = "/content/drive/MyDrive/ulasan_aplikasi.csv"

In [ ]:
# CELL 3: CONFIG & IMPORTS — FULL GPU T4 OPTIMIZATION

import re
import json
import time
import itertools
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import copy
import emoji
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss
import torch.nn.functional as F
from transformers import EarlyStoppingCallback
import nltk
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('vader_lexicon', quiet=True)

# Seeds set identically in numpy, torch, and random for reproducible experiments.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Select GPU if available; all models and batches are moved to this DEVICE.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# FULL GPU T4 OPTIMIZATION (16GB VRAM, Compute Capability 7.5)

# 1. TF32 matmul — T4 supports Tensor Float 32 for acceleration
# TF32 accelerates matrix multiplication operations on NVIDIA GPU without significant overhead.
# Suitable for deep learning training as slight precision loss is usually insignificant.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# 2. cuDNN benchmark — auto-tune best kernel for fixed input sizes
# benchmark=True asks cuDNN to find the fastest kernel for relatively stable input shapes.
# deterministic=False chosen as trade-off: results may vary slightly, but training is faster.
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False  # non-deterministic = faster

# 3. Precision hint for matmul
# Matmul precision hint maintains high performance when PyTorch selects float32 operation implementation.
torch.set_float32_matmul_precision('high')

# 4. CPU thread optimization for data preprocessing
#    T4 Colab typically has 2 vCPU
# Limit CPU threads to prevent preprocessing/DataLoader from competing with Colab runtime resources.
# try/except used because thread count cannot always be changed after runtime is active.
try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass
try:
    torch.set_num_threads(2)
except RuntimeError:
    pass

# 5. Clear GPU cache before starting
# Empty CUDA allocator cache from previous executions for cleaner memory baseline.
torch.cuda.empty_cache()

# 6. Display GPU info
# GPU info printed at start to confirm notebook uses expected accelerator.
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Cores: {torch.cuda.get_device_properties(0).multi_processor_count * 128}")
    print(f"   Compute Capability: {torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}")
    print(f"   PyTorch: {torch.__version__}")
else:
    print("GPU not available, using CPU")

# Label mapping made explicit to ensure class order consistency across training, evaluation, and inference.
# ID2LABEL is the reverse to convert numeric predictions back to sentiment names.
LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}

# All experiment results collected in this list so final table can be created from single source.
experiment_results = []

# 7. DataLoader optimization
# pin_memory accelerates CPU→GPU transfer; num_workers=2 fits typical Colab Free/T4 limits.
DL_KWARGS = {'num_workers': 2, 'pin_memory': True} if torch.cuda.is_available() else {}

# Helper: standardized banner & print functions
# Following helpers maintain consistent log format so progress across models is easy to compare.
def print_banner(title, device=DEVICE):
    """Function to print device and VRAM information.
    
    Args:
        title (str): Title to be displayed
        device (torch.device): Device being used (default: DEVICE)
    """
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram     = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "-"
    print(f"\n{title}")
    print(f"Device : {gpu_name} ({device})")
    print(f"VRAM   : {vram}")

# Used to check model size that is actually trainable, not all parameters.
def count_parameters(model):
    """Calculate number of trainable parameters in model.
    
    Args:
        model (nn.Module): PyTorch model
        
    Returns:
        int: Total number of trainable parameters
    """
    return sum(p.numel() for p in model.parameters())

# Per-epoch summary made compact so changes in loss, metrics, time, and LR are quick to see.
def print_epoch(epoch, total, loss, metric_name, metric_val, elapsed, lr=None):
    """Print training epoch information in formatted manner.
    
    Args:
        epoch (int): Current epoch number
        total (int): Total number of epochs
        loss (float): Current loss value
        metric_name (str): Name of metric being used
        metric_val (float): Current metric value
        elapsed (float): Time elapsed in seconds
        lr (float, optional): Current learning rate
    """
    lr_str = f" | LR: {lr:.6f}" if lr is not None else ""
    print(f"  Epoch [{epoch:>2}/{total}] | Loss: {loss:.4f} | "
          f"{metric_name}: {metric_val:.4f} | Time: {elapsed:.1f}s{lr_str}")

# Best run summary makes auditing hyperparameters selected from grid search easier.
def print_best(model_name, best_params, metric_name, metric_val, total_time):
    """Print information of best model after training completes.
    
    Args:
        model_name (str): Model name
        best_params (dict): Best parameters found
        metric_name (str): Name of metric being used
        metric_val (float): Best metric value
        total_time (float): Total training time in seconds
    """
    print(f"\n{model_name} — Best {metric_name}: {metric_val:.4f}")
    print(f"    Params : {best_params}")
    print(f"    Total  : {total_time:.1f}s\n")

In [ ]:
# CELL 4: LOAD, CLEAN, LABEL + HYBRID SAMPLING

t0_load = time.time()

# Loader accepts two possible review column names for compatibility with different datasets.
# NaN and duplicates dropped from start so automatic labels are not biased by repeated data.
def load_dataset(path):
    df = pd.read_csv(path)
    target_col = 'content' if 'content' in df.columns else 'Review'
    clean_df = df[[target_col]].dropna().drop_duplicates()
    clean_df.columns = ['content']
    return clean_df.reset_index(drop=True)

# Cleaning kept light: social/media noise removed, but main sentence structure retained.
def clean_text_light(text):
    """Clean review text by removing irrelevant elements.
    
    Args:
        text (str): Review text to be cleaned
        
    Returns:
        str: Cleaned text with mentions, hashtags, retweets, URLs, and emoji removed
    """
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    # Remove decorative characters/keyboard emoticons that might affect encoding
    text = re.sub(r'[═║╔╚╗╝╠╣╦╩⭐★✅✔⚠️📊📈⏹─━│┃▸■▪▫▶◆◇]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# VADER used as lexicon-based weak labeler; suitable for baseline sentiment without manual labels.
sia = SentimentIntensityAnalyzer()

# Small thresholds near zero separate neutral text from positive/negative based on compound score.
def label_three_class(text, pos_thr=0.05, neg_thr=-0.05):
    """Label text into three sentiment classes based on VADER score.
    
    Args:
        text (str): Text to be labeled
        pos_thr (float, optional): Threshold for positive class. Default 0.05.
        neg_thr (float, optional): Threshold for negative class. Default -0.05.
        
    Returns:
        tuple: (score, label) where score is sentiment score and label is
               'positive', 'neutral', or 'negative'
    """
    score = sia.polarity_scores(text)['compound']
    if score >= pos_thr:   return score, 'positive'
    elif score <= neg_thr: return score, 'negative'
    return score, 'neutral'

# Label pipeline: load → clean for model input → VADER score → numeric label mapping.
df_raw = load_dataset(CSV_PATH)
df_raw['text_clean'] = df_raw['content'].apply(clean_text_light)
scores_labels = df_raw['content'].apply(label_three_class)
df_raw['polarity_score'] = scores_labels.apply(lambda x: x[0])
df_raw['polarity']       = scores_labels.apply(lambda x: x[1])
df_raw['label']          = df_raw['polarity'].map(LABEL_MAP)

print(f"Total data: {df_raw.shape[0]}  |  Load: {time.time()-t0_load:.2f}s")
print("\nINITIAL Distribution:")
print(df_raw['polarity'].value_counts())

# NEW DATA STRATEGY: HYBRID SAMPLING
# Undersample majority + Oversample minority to same target
# Target: 3000 per class (not 1195 as before)
# This provides 3x more data than before

# Hybrid sampling balances classes without discarding all majority or duplicating raw minority.
# Target per class made same so loss and metrics not dominated by most frequent label.
def hybrid_sampling(df, label_col='label', target_per_class=3000, seed=42):
    """
    Hybrid sampling:
    - Class > target → undersample
    - Class < target → oversample (duplicate + light augmentation)
    - Class = target → keep as is
    """
    # Local RNG keeps sampling/augmentation results reproducible without affecting global seed.
    rng = np.random.RandomState(seed)
    frames = []

    for label in sorted(df[label_col].unique()):
        subset = df[df[label_col] == label]
        n = len(subset)

        # Majority class cut to target so dataset balanced and training faster.
        if n >= target_per_class:
            # Undersample
            sampled = subset.sample(n=target_per_class, random_state=seed)
        # Minority class expanded via resampling as original data hasn't reached target.
        else:
            # Oversample: duplicate + light augmentation
            sampled = subset.copy()
            n_need = target_per_class - n
            extra_idx = rng.choice(subset.index, size=n_need, replace=True)
            extra = subset.loc[extra_idx].copy()

            # Light augmentation: random word deletion (10% words deleted)
            # Minority duplicates given light variation so model doesn't just memorize same text.
            augmented_texts = []
            for text in extra['text_clean']:
                words = text.split()
                if len(words) > 3:
                    n_del = max(1, int(len(words) * 0.1))
                    for _ in range(n_del):
                        if len(words) > 2:
                            idx = rng.randint(0, len(words))
                            words.pop(idx)
                augmented_texts.append(' '.join(words))
            extra['text_clean'] = augmented_texts
            sampled = pd.concat([sampled, extra], ignore_index=True)

        frames.append(sampled)

    # Final shuffle mixes all classes after sampling so training batches aren't sorted by label.
    result = pd.concat(frames, ignore_index=True)
    result = result.sample(frac=1, random_state=seed).reset_index(drop=True)
    return result

# This target becomes main knob for trading training time vs examples per class.
TARGET_PER_CLASS = 3000  # 3x more than before (1195)
df_raw = hybrid_sampling(df_raw, target_per_class=TARGET_PER_CLASS)

print(f"\nDistribution AFTER hybrid sampling (target={TARGET_PER_CLASS}/class):")
print(df_raw['polarity'].value_counts())
print(f"Total: {df_raw.shape[0]}")

In [ ]:
# CELL 5: HELPER UTILITIES + TEXT AUGMENTATION + EARLY STOPPING

# Stratified split ensures class ratios remain consistent across train/val/test.
def make_split(df, text_col, label_col, test_ratio, val_ratio_of_train=0.15, seed=RANDOM_STATE):
    """Split dataset into train, validation, and test sets with stratification.
    
    Args:
        df (pd.DataFrame): DataFrame containing data
        text_col (str): Column name containing text
        label_col (str): Column name containing labels
        test_ratio (float): Proportion of data for test set
        val_ratio_of_train (float, optional): Proportion of data for validation set from train set. Default 0.15.
        seed (int, optional): Random seed for reproducibility. Default RANDOM_STATE.
        
    Returns:
        tuple: (X_train, X_val, X_test, y_train, y_val, y_test) - dataset split
    """
    # Test separated first so it doesn't influence hyperparameter selection.
    X_trainfull, X_test, y_trainfull, y_test = train_test_split(
        df[text_col], df[label_col], test_size=test_ratio,
        stratify=df[label_col], random_state=seed
    )
    # Validation taken from trainfull for early stopping and grid search.
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainfull, y_trainfull, test_size=val_ratio_of_train,
        stratify=y_trainfull, random_state=seed
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# Class weights still calculated even with balanced sampling as protection if split distribution changes.
def get_class_weights(y_train_labels):
    """Calculate class weights to handle class imbalance.
    
    This function uses sklearn's 'balanced' method to calculate class weights
    inversely proportional to class frequency. Weights used in loss function
    to give higher penalty to minority classes.
    
    Args:
        y_train_labels (array-like): Labels from training set
        
    Returns:
        torch.Tensor: Tensor of class weights usable in loss function
    """
    # Balanced weights give higher penalty for classes with fewer samples in training split.
    weights = compute_class_weight('balanced',
                                   classes=np.array(list(LABEL_MAP.values())),
                                   y=y_train_labels)
    return torch.tensor(weights, dtype=torch.float)

# TEXT AUGMENTATION
# Augmentation intentionally simple to keep sentiment labels valid while adding word order variation.
def augment_text(text, rng=None):
    """Perform text augmentation with random deletion and random swap.
    
    This augmentation technique increases training data variety by:
    - Random deletion: randomly remove 10% of words
    - Random swap: randomly swap two words positions
    
    Args:
        text (str): Text to be augmented
        rng (np.random.RandomState, optional): Random number generator. Default None.
        
    Returns:
        str: Augmented text
    """
    # Optional RNG simplifies reproducibility control when augmentation used in Dataset.
    if rng is None:
        rng = np.random.RandomState()
    words = text.split()
    if len(words) <= 3:
        return text

    # Random deletion mimics variation in short/incomplete reviews without changing most words.
    # Random deletion (10%)
    if rng.random() < 0.5:
        n_del = max(1, int(len(words) * 0.1))
        for _ in range(n_del):
            if len(words) > 2:
                idx = rng.randint(0, len(words))
                words.pop(idx)

    # Random swap trains model to not be too sensitive to positions of non-critical words.
    # Random swap (5%)
    if rng.random() < 0.5 and len(words) > 2:
        i, j = rng.choice(len(words), size=2, replace=False)
        words[i], words[j] = words[j], words[i]

    return ' '.join(words)

# EARLY STOPPING
# EarlyStopping saves best state so final model is not just last epoch.
class EarlyStopping:
    # patience determines epoch tolerance without improvement; min_delta avoids treating small noise as improvement.
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.best_state = None
        self.should_stop = False

    # Called after validation; if score improves, model snapshot saved in memory.
    def __call__(self, score, model):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    # After loop stops, best weights reloaded before test evaluation.
    def load_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

# LABEL SMOOTHING LOSS
# Label smoothing prevents model from being overconfident on weak labels from VADER.
class LabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes=3, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.weight = weight

    # One-hot targets softened then combined with class weight per sample.
    def forward(self, pred, target):
        log_prob = F.log_softmax(pred, dim=-1)
        one_hot = torch.zeros_like(pred).scatter(1, target.unsqueeze(1), 1)
        one_hot = one_hot * (1 - self.smoothing) + self.smoothing / self.num_classes
        loss = -(one_hot * log_prob).sum(dim=-1)
        if self.weight is not None:
            w = self.weight[target]
            loss = loss * w
        return loss.mean()

def log_result(model_name, split_info, feature_info,
               y_train_true, y_train_pred, y_test_true, y_test_pred, best_params):
    result = {
        "Model": model_name, "Split": split_info, "Feature": feature_info,
        "Best Params": str(best_params),
        "Train Accuracy": accuracy_score(y_train_true, y_train_pred),
        "Test Accuracy": accuracy_score(y_test_true, y_test_pred),
        "Precision": precision_score(y_test_true, y_test_pred, average='macro', zero_division=0),
        "Recall": recall_score(y_test_true, y_test_pred, average='macro', zero_division=0),
        "F1-Score": f1_score(y_test_true, y_test_pred, average='macro', zero_division=0),
    }
    experiment_results.append(result)
    return result

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(LABEL_MAP.keys()),
                yticklabels=list(LABEL_MAP.keys()))
    plt.title(title); plt.xlabel('Prediction'); plt.ylabel('Actual')
    plt.tight_layout(); plt.show()

def build_vocab(texts, min_freq=2):
    counter = Counter()
    for t in texts:
        counter.update(word_tokenize(t.lower()))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def text_to_ids(text, vocab, max_len):
    tokens = word_tokenize(text.lower())[:max_len]
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids

In [ ]:
# CELL 6: MODEL A — TextCNN, Split 80:20
# (ENHANCED: Early Stop + LR Scheduler + Label Smoothing + Augmentation + Higher Dropout)

torch.cuda.empty_cache()
# Split 80:20 provides more training data for CNN model learning embeddings from scratch.
SPLIT_TEST_RATIO_A = 0.20

print_banner("MODEL A — TextCNN  (Split 80:20)")

X_tr_A, X_val_A, X_te_A, y_tr_A, y_val_A, y_te_A = make_split(
    df_raw, 'text_clean', 'label', test_ratio=SPLIT_TEST_RATIO_A,
    val_ratio_of_train=0.15  # larger validation set
)

# MAX_LEN limits token count so batches are dense and convolutions remain efficient on GPU.
MAX_LEN_A = 50  # slightly longer
vocab_A = build_vocab(X_tr_A, min_freq=2)
VOCAB_SIZE_A = len(vocab_A)
print(f"  Vocab: {VOCAB_SIZE_A} | Train: {len(X_tr_A)} | Val: {len(X_val_A)} | Test: {len(X_te_A)}")

# TextCNN Dataset converts text to vocabulary indices and applies augmentation only during training.
# TextCNN uses multiple kernel sizes to capture phrase patterns from short to moderately long.
class TextCNNDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len, augment=False):
        self.texts  = list(texts)
        self.vocab  = vocab
        self.max_len = max_len
        self.labels = labels.values if hasattr(labels, 'values') else labels
        self.augment = augment
        self.rng = np.random.RandomState(RANDOM_STATE)
        # Pre-tokenize if not augmenting
        # Without augmentation, tokenization cached once to reduce overhead during validation/test.
        if not augment:
            self.ids = [text_to_ids(t, vocab, max_len) for t in texts]
        else:
            self.ids = None

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        # Probabilistic augmentation keeps most batch with original text.
        if self.augment and self.rng.random() < 0.3:  # 30% chance augment
            text = augment_text(self.texts[idx], self.rng)
        else:
            text = self.texts[idx]
        ids = text_to_ids(text, self.vocab, self.max_len)
        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, num_filters=100,
                 kernel_sizes=(2, 3, 4, 5), num_classes=3, dropout=0.4):
        super().__init__()
        # Trainable embedding learns domain-specific representation for game reviews.
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Parallel Conv1d works like n-gram detectors with different window sizes.
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    # Forward: embedding → convolution → global max-pooling → classifier.
    def forward(self, x):
        emb = self.dropout(self.embedding(x)).permute(0, 2, 1)  # dropout on embedding
        # ReLU preserves strongly active phrase features for each filter.
        conv_outs = [torch.relu(conv(emb)) for conv in self.convs]
        # Max-pooling takes strongest signal from each filter independent of token position.
        pooled = [torch.max(c, dim=2)[0] for c in conv_outs]
        concat = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(concat)

# Training function isolates one hyperparameter combination for easy grid search execution.
def train_textcnn(num_filters, dropout, lr, epochs=15, patience=4):
    """Train TextCNN model with specified hyperparameters.
    
    This function performs training loop for TextCNN model with features:
    - Label smoothing loss for regularization
    - Class weights for handling imbalance
    - Cosine annealing learning rate scheduler
    - Gradient scaling for mixed precision training
    - Early stopping to prevent overfitting
    
    Args:
        num_filters (int): Number of filters per convolution
        dropout (float): Dropout rate
        lr (float): Learning rate
        epochs (int, optional): Maximum number of epochs. Default 15.
        patience (int, optional): Patience for early stopping. Default 4.
        
    Returns:
        tuple: (model, best_val_acc) - best model and best validation accuracy
    """
    model = TextCNN(VOCAB_SIZE_A, num_filters=num_filters,
                    dropout=dropout, kernel_sizes=(2,3,4,5)).to(DEVICE)
    # Class weights calculated from train split so loss reflects true training data distribution.
    class_weights = get_class_weights(y_tr_A.values).to(DEVICE)

    # Label Smoothing + Weighted
    criterion = LabelSmoothingLoss(num_classes=3, smoothing=0.1, weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)  # larger weight decay

    # Cosine Annealing LR Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=3, T_mult=2, eta_min=1e-5
    )

    # GradScaler stabilizes mixed precision so training faster without gradient underflow.
    scaler = torch.amp.GradScaler('cuda')
    early_stop = EarlyStopping(patience=patience, min_delta=0.002)

    # Train batch smaller due to backward pass; val larger since inference only.
    train_loader = DataLoader(
        TextCNNDataset(X_tr_A, y_tr_A, vocab_A, MAX_LEN_A, augment=True),
        batch_size=64, shuffle=True, **DL_KWARGS
    )
    val_loader = DataLoader(
        TextCNNDataset(X_val_A, y_val_A, vocab_A, MAX_LEN_A, augment=False),
        batch_size=128, **DL_KWARGS
    )

    best_val_acc = 0
    # Epoch loop stops early if validation no longer improves.
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        t_ep = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # Gradient clipping limits extreme updates common with mixed precision.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * xb.size(0)

        # Cosine scheduler reduces/increases LR periodically so optimizer doesn't easily stagnate.
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        # Validation
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad(), torch.amp.autocast('cuda'):
            for xb, yb in val_loader:
                preds = torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(yb.numpy())

        avg_loss = running_loss / len(train_loader.dataset)
        val_acc = accuracy_score(val_true, val_preds)
        print_epoch(epoch, epochs, avg_loss, "Val Acc", val_acc,
                    time.time()-t_ep, lr=current_lr)

        # Early Stopping
        early_stop(val_acc, model)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if early_stop.should_stop:
            print(f"  Early stopping at epoch {epoch}")
            break

    early_stop.load_best(model)
    return model, best_val_acc

# --- Grid Search (more focused) ---
# Grid kept small and focused so exploration completes realistically on Colab T4.
param_grid_A = {
    'num_filters': [100, 150],
    'dropout': [0.3, 0.4],
    'lr': [5e-4, 1e-3]
}
best_val_acc_A, best_model_A, best_params_A = -1, None, None
t_start_A = time.time()

# Each combination trained independently; validation metrics determine model for test.
for nf, do, lr in itertools.product(
    param_grid_A['num_filters'], param_grid_A['dropout'], param_grid_A['lr']
):
    print(f"\n num_filters={nf}, dropout={do}, lr={lr}")
    model, val_acc = train_textcnn(nf, do, lr, epochs=15, patience=4)
    if val_acc > best_val_acc_A:
        best_val_acc_A, best_model_A = val_acc, model
        best_params_A = {'num_filters': nf, 'dropout': do, 'lr': lr}

print_best("Model A: TextCNN", best_params_A, "Val Acc", best_val_acc_A,
           time.time()-t_start_A)

# Prediction helper reused by TextCNN, BiLSTM, and ensemble so evaluation consistent.
def predict_dl_model(model, dataset_cls, texts, vocab=None, max_len=None, w2v=None):
    dummy_labels = pd.Series(np.zeros(len(texts)))
    if vocab is not None:
        loader = DataLoader(dataset_cls(texts, dummy_labels, vocab, max_len, augment=False),
                            batch_size=128, **DL_KWARGS)
    else:
        loader = DataLoader(dataset_cls(texts, dummy_labels, w2v),
                            batch_size=128, **DL_KWARGS)
    model.eval()
    preds = []
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for xb, _ in loader:
            preds.extend(torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1).cpu().numpy())
    return preds

y_pred_train_A = predict_dl_model(best_model_A, TextCNNDataset, X_tr_A, vocab=vocab_A, max_len=MAX_LEN_A)
y_pred_test_A  = predict_dl_model(best_model_A, TextCNNDataset, X_te_A, vocab=vocab_A, max_len=MAX_LEN_A)

res_A = log_result("Model A: TextCNN",
                    f"{int((1-SPLIT_TEST_RATIO_A)*100)}:{int(SPLIT_TEST_RATIO_A*100)}",
                    "Trainable Embedding + Augmentation + Label Smoothing",
                    y_tr_A, y_pred_train_A, y_te_A, y_pred_test_A, best_params_A)
print(f"  Test Acc: {res_A['Test Accuracy']:.4f} | F1: {res_A['F1-Score']:.4f}")
plot_confusion(y_te_A, y_pred_test_A, "Confusion Matrix — Model A (TextCNN)")

In [7]:
# CELL 7: MODEL B — BiLSTM + GloVe + ATTENTION, Split 70:30

torch.cuda.empty_cache()
# Split 70:30 tests BiLSTM on larger test set to see sequence generalization.
SPLIT_TEST_RATIO_B = 0.30

print_banner("MODEL B — BiLSTM + GloVe + Attention (Split 70:30)")

X_tr_B, X_val_B, X_te_B, y_tr_B, y_val_B, y_te_B = make_split(
    df_raw, 'text_clean', 'label', test_ratio=SPLIT_TEST_RATIO_B,
    val_ratio_of_train=0.15
)

# Token length same as TextCNN for fair non-transformer model comparison.
MAX_LEN_B = 50
EMBED_DIM_B = 100
vocab_B = build_vocab(X_tr_B, min_freq=2)
VOCAB_SIZE_B = len(vocab_B)
print(f"  Vocab: {VOCAB_SIZE_B} | Train: {len(X_tr_B)} | Val: {len(X_val_B)} | Test: {len(X_te_B)}")

# GloVe provides initial embeddings from large corpus so BiLSTM doesn't learn representations from scratch.
def load_glove_embeddings(glove_path, vocab, embed_dim):
    # Kata yang tidak ada di GloVe diberi nilai acak kecil; PAD dibuat nol agar tidak membawa sinyal.
    embedding_matrix = np.random.uniform(-0.05, 0.05, (len(vocab), embed_dim)).astype(np.float32)
    embedding_matrix[vocab['<PAD>']] = np.zeros(embed_dim)
    found = 0
    # Load only vectors for words in vocabulary to keep memory efficient.
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
                found += 1
    print(f"  GloVe coverage: {found}/{len(vocab)} ({found/len(vocab)*100:.1f}%)")
    return torch.tensor(embedding_matrix, dtype=torch.float32)

glove_embedding_matrix = load_glove_embeddings(GLOVE_PATH, vocab_B, EMBED_DIM_B)

# BiLSTM Dataset uses vocabulary same concept as TextCNN with augmentation only for train.
class BiLSTMDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN_B, augment=False):
        self.texts = list(texts)
        self.vocab = vocab
        self.max_len = max_len
        self.labels = labels.values if hasattr(labels, 'values') else labels
        self.augment = augment
        self.rng = np.random.RandomState(RANDOM_STATE)
        # Cache ids untuk mode non-augment mengurangi biaya tokenisasi berulang saat evaluasi.
        if not augment:
            self.ids = [text_to_ids(t, vocab, max_len) for t in texts]
        else:
            self.ids = None

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        # Probabilitas 30% memberi regularisasi tanpa membuat distribusi teks terlalu berbeda.
        if self.augment and self.rng.random() < 0.3:
            text = augment_text(self.texts[idx], self.rng)
        else:
            text = self.texts[idx]
        ids = text_to_ids(text, self.vocab, self.max_len)
        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

# BiLSTM + Self-Attention
# BiLSTM reads left-right context; attention selects tokens/fragments most relevant for sentiment.
class AttentionBiLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_layers=1,
                 num_classes=3, dropout=0.4, freeze_embed=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        # Embedding can be frozen or fine-tuned to trade stability vs domain adaptation.
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=freeze_embed, padding_idx=0)
        self.embed_drop = nn.Dropout(0.2)
        # Bidirectional LSTM captures dependencies before and after target word.
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)

        # Self-Attention
        # Attention generates weights per timestep to summarize sequence into single context vector.
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False)
        )

        # MLP Head with dropout converts context vector to three class logits.
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, num_classes)
        )

    # Forward: GloVe embedding → BiLSTM → attention pooling → classifier.
    def forward(self, x):
        emb = self.embed_drop(self.embedding(x))
        lstm_out, _ = self.lstm(emb)  # (batch, seq, hidden*2)

        # Attention weights
        attn_weights = self.attention(lstm_out).squeeze(-1)  # (batch, seq)
        attn_weights = F.softmax(attn_weights, dim=-1)
        # Weighted sum memberi representasi kalimat yang fokus pada token berbobot tinggi.
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_out).squeeze(1)

        return self.fc(context)

# Fungsi training BiLSTM dibuat paralel dengan TextCNN agar strategi regularisasi konsisten.
def train_bilstm(hidden_dim, num_layers, lr, freeze_embed, epochs=12, patience=4):
    """Melatih model BiLSTM dengan self-attention menggunakan hyperparameter tertentu.
    
    This function performs training loop for BiLSTM model with features:
    - Label smoothing loss for regularization
    - Class weights to handle imbalance
    - Cosine annealing learning rate scheduler
    - Gradient scaling for mixed precision training
    - Early stopping to prevent overfitting
    
    Args:
        hidden_dim (int): LSTM hidden state dimension
        num_layers (int): Number of LSTM layers
        lr (float): Learning rate
        freeze_embed (bool): Whether to freeze embedding layer
        epochs (int, optional): Maximum number of epochs. Default 12.
        patience (int, optional): Patience for early stopping. Default 4.
        
    Returns:
        tuple: (model, best_val_acc) - best model and best validation accuracy
    """
    model = AttentionBiLSTM(glove_embedding_matrix, hidden_dim=hidden_dim,
                            num_layers=num_layers, freeze_embed=freeze_embed,
                            dropout=0.4).to(DEVICE)
    class_weights = get_class_weights(y_tr_B.values).to(DEVICE)
    criterion = LabelSmoothingLoss(num_classes=3, smoothing=0.1, weight=class_weights)

    # AdamW + weight decay helps regularize LSTM parameters and classification head.
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=3, T_mult=2, eta_min=1e-5
    )
    scaler = torch.amp.GradScaler('cuda')
    early_stop = EarlyStopping(patience=patience, min_delta=0.002)

    # Train uses augmentation; validation uses original text so metrics reflect real data.
    train_loader = DataLoader(
        BiLSTMDataset(X_tr_B, y_tr_B, vocab_B, MAX_LEN_B, augment=True),
        batch_size=64, shuffle=True, **DL_KWARGS
    )
    val_loader = DataLoader(
        BiLSTMDataset(X_val_B, y_val_B, vocab_B, MAX_LEN_B, augment=False),
        batch_size=128, **DL_KWARGS
    )

    best_val_acc = 0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        t_ep = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # Clipping important for RNN as gradients can spike on certain sequences.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * xb.size(0)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad(), torch.amp.autocast('cuda'):
            for xb, yb in val_loader:
                preds = torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(yb.numpy())

        avg_loss = running_loss / len(train_loader.dataset)
        val_acc = accuracy_score(val_true, val_preds)
        print_epoch(epoch, epochs, avg_loss, "Val Acc", val_acc,
                    time.time()-t_ep, lr=current_lr)

        early_stop(val_acc, model)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if early_stop.should_stop:
            print(f"  Early stopping di epoch {epoch}")
            break

    early_stop.load_best(model)
    return model, best_val_acc

# --- Grid Search ---
# BiLSTM grid focused on number of layers and LR since embedding already helped by GloVe.
param_grid_B = {
    'hidden_dim': [128],
    'num_layers': [1, 2],
    'lr': [5e-4, 1e-3],
    'freeze_embed': [False]
}
best_val_acc_B, best_model_B, best_params_B = -1, None, None
t_start_B = time.time()

# Best combination selected from validation accuracy before test set evaluation.
for hd, nl, lr, fe in itertools.product(
    param_grid_B['hidden_dim'], param_grid_B['num_layers'],
    param_grid_B['lr'], param_grid_B['freeze_embed']
):
    print(f"\n hidden={hd}, layers={nl}, lr={lr}, freeze={fe}")
    model, val_acc = train_bilstm(hd, nl, lr, fe, epochs=12, patience=4)
    if val_acc > best_val_acc_B:
        best_val_acc_B, best_model_B = val_acc, model
        best_params_B = {'hidden_dim': hd, 'num_layers': nl, 'lr': lr, 'freeze_embed': fe}

print_best("Model B: BiLSTM+Attn+GloVe", best_params_B, "Val Acc", best_val_acc_B,
           time.time()-t_start_B)

y_pred_train_B = predict_dl_model(best_model_B, BiLSTMDataset, X_tr_B, vocab=vocab_B, max_len=MAX_LEN_B)
y_pred_test_B  = predict_dl_model(best_model_B, BiLSTMDataset, X_te_B, vocab=vocab_B, max_len=MAX_LEN_B)

res_B = log_result("Model B: BiLSTM+Attn+GloVe",
                    f"{int((1-SPLIT_TEST_RATIO_B)*100)}:{int(SPLIT_TEST_RATIO_B*100)}",
                    "GloVe + Self-Attention + Label Smoothing",
                    y_tr_B, y_pred_train_B, y_te_B, y_pred_test_B, best_params_B)
print(f"  Test Acc: {res_B['Test Accuracy']:.4f} | F1: {res_B['F1-Score']:.4f}")
plot_confusion(y_te_B, y_pred_test_B, "Confusion Matrix — Model B (BiLSTM+Attn)")

In [8]:
# CELL 8: Model C — RoBERTa Fine-tuning
# Target: Colab Free GPU T4 (16 GB VRAM, ~10 TFLOPS FP16)

import math, gc, copy

# Clear cache and garbage collector as fine-tuning transformers very sensitive to VRAM.
torch.cuda.empty_cache()
gc.collect()

# Split 75:25 serves as main reference for RoBERTa and final ensemble.
SPLIT_TEST_RATIO_C = 0.25
print_banner("MODEL C — RoBERTa Fine-tuned (Split 75:25)")

X_tr_C, X_val_C, X_te_C, y_tr_C, y_val_C, y_te_C = make_split(
    df_raw, 'text_clean', 'label',
    test_ratio=SPLIT_TEST_RATIO_C,
    val_ratio_of_train=0.15
)

# CardiffNLP model chosen as it's pretrained on Twitter/social data similar to short review style.
MODEL_NAME_C = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer_C  = AutoTokenizer.from_pretrained(MODEL_NAME_C)

# Batch tokenization + measure actual length
# Tokenization made as function so train/val/test use same padding and truncation.
def tokenize_texts(texts, max_length=128):
    # padding=max_length membuat tensor berukuran tetap sehingga batching di Trainer lebih sederhana.
    return tokenizer_C(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

# MAX_LEN determined from p95 token length so majority text intact without wasting VRAM.
sample_lengths = [len(tokenizer_C.encode(t)) for t in X_tr_C[:500]]
p95_len = int(np.percentile(sample_lengths, 95))
MAX_LEN_C = min(128, max(64, p95_len + 8))
print(f"  p95 token length = {p95_len} → max_length = {MAX_LEN_C}")

t_tok = time.time()
# Encoding dilakukan sekali di awal agar Trainer tidak mengulang tokenisasi pada setiap epoch.
train_enc_C = tokenize_texts(X_tr_C, MAX_LEN_C)
val_enc_C   = tokenize_texts(X_val_C, MAX_LEN_C)
test_enc_C  = tokenize_texts(X_te_C, MAX_LEN_C)
print(f"  Tokenization: {time.time()-t_tok:.1f}s")
print(f"  Train: {len(X_tr_C)} | Val: {len(X_val_C)} | Test: {len(X_te_C)}")

# Dataset (tidak berubah, sudah efisien)
# HuggingFace Dataset returns dict per Trainer format: input_ids, attention_mask, labels.
class HFDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(
            labels.values if hasattr(labels, 'values') else labels,
            dtype=torch.long
        )
    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }
    def __len__(self):
        return len(self.labels)

train_dataset_C = HFDataset(train_enc_C, y_tr_C)
val_dataset_C   = HFDataset(val_enc_C,   y_val_C)
test_dataset_C  = HFDataset(test_enc_C,  y_te_C)

# Class weights used in custom Trainer to keep loss sensitive to each class.
class_weights_C = get_class_weights(y_tr_C.values)

# Mean-Pooling Head: lebih stabil dari [CLS] saja
# Mean pooling memanfaatkan semua token non-padding, bukan hanya token awal, sehingga lebih stabil untuk ulasan.
class RoBERTaMeanPoolHead(nn.Module):
    """Arsitektur RoBERTa dengan mean-pooling head untuk klasifikasi.
    
    Model uses pretrained RoBERTa encoder with custom classification head:
    - Mean-pooling from hidden states to get sentence representation
    - Dropout for regularization
    - Linear layer for classification
    - Xavier initialization for training stability
    
    Args:
        base_model (AutoModel): Pretrained RoBERTa model
        num_labels (int, optional): Jumlah kelas. Default 3.
        dropout_rate (float, optional): Tingkat dropout. Default 0.2.
    """
    def __init__(self, base_model, num_labels=3, dropout_rate=0.2):
        super().__init__()
        # Only RoBERTa encoder used; built-in classification head replaced with simple head.
        self.roberta    = base_model.roberta          # encoder saja
        self.dropout    = nn.Dropout(dropout_rate)     # dropout eksplisit
        self.classifier = nn.Linear(768, num_labels)
        # initialize head so it doesn't start from large random values
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):
        """Forward pass model RoBERTaMeanPoolHead.
        
        Args:
            input_ids (torch.Tensor): Input tensor dengan shape (batch_size, seq_len)
            attention_mask (torch.Tensor): Attention mask dengan shape (batch_size, seq_len)
            labels (torch.Tensor, optional): Label ground truth
            **kw: Argumen tambahan
            
        Returns:
            dict: Dictionary berisi 'loss' dan 'logits' jika labels diberikan,
                  atau hanya 'logits' jika tidak ada labels
        """
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # attention_mask ensures padding tokens not included in representation average.
        hidden = outputs.last_hidden_state                       # (B, L, 768)
        mask_exp = attention_mask.unsqueeze(-1).float()          # (B, L, 1)
        pooled = (hidden * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        pooled = self.dropout(pooled)                            # (B, 768)
        logits = self.classifier(pooled)                         # (B, num_labels)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)               # placeholder
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


# Layer-wise Learning Rate Decay (LLRD)
# Lower layers mature → small LR; head → large LR
# LLRD gives different LR per layer: lower layers kept stable, upper layers/head adapt faster.
def build_llrd_optimizer(model, base_lr=2e-5, weight_decay=0.03, lr_decay=0.9):
    # Bias dan LayerNorm tidak diberi weight decay agar statistik normalisasi tidak terdistorsi.
    no_decay_keys = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    param_groups  = []

    # Each parameter placed in own param group so LR multiplier can be precise.
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # determine multiplier per component
        if "classifier" in name:
            lr_mult = 10.0                          # head → largest
        elif "pooler" in name:
            lr_mult = 5.0
        elif "encoder.layer." in name:
            layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
            lr_mult = lr_decay ** (11 - layer_num)  # layer 11→1.0, layer 0→≈0.28
        else:
            lr_mult = lr_decay ** 12                # embeddings → smallest

        param_groups.append({
            "params":       [param],
            "lr":           base_lr * lr_mult,
            "weight_decay": 0.0 if any(nd in name for nd in no_decay_keys) else weight_decay,
        })

    return torch.optim.AdamW(param_groups)


# Trainer with Label Smoothing + Class Weights + LLRD
# Custom Trainer combines label smoothing, class weight, and LLRD optimizer in HuggingFace API.
class WeightedLabelSmoothingTrainer(Trainer):
    def __init__(self, *args, class_weights=None, smoothing=0.05,
                 base_lr=2e-5, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.smoothing     = smoothing
        self.base_lr       = base_lr                     # untuk LLRD

    # Override optimizer → use LLRD
    # Override ensures Trainer uses LLRD optimizer, not default AdamW.
    def create_optimizer(self):
        if self.optimizer is None:
            self.optimizer = build_llrd_optimizer(
                self.model,
                base_lr=self.base_lr,
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer

    # Custom loss replaces standard cross-entropy so weak labels not learned too rigidly.
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits if hasattr(outputs, 'logits') else outputs["logits"]
        num_labels = logits.size(-1)

        log_prob = F.log_softmax(logits, dim=-1)
        one_hot  = torch.zeros_like(logits).scatter(1, labels.unsqueeze(1), 1)
        one_hot  = one_hot * (1 - self.smoothing) + self.smoothing / num_labels

        # Weight per sample diambil dari label asli sehingga kelas minoritas mendapat penalti lebih besar.
        if self.class_weights is not None:
            w    = self.class_weights.to(logits.device)[labels]
            loss = -(one_hot * log_prob).sum(dim=-1) * w
        else:
            loss = -(one_hot * log_prob).sum(dim=-1)
        loss = loss.mean()

        return (loss, outputs) if return_outputs else loss


# Macro F1 monitored as more informative than accuracy when performance imbalanced across classes.
def compute_metrics(eval_pred):
    """Menghitung metrik evaluasi untuk model.
    
    Args:
        eval_pred (tuple): Tuple berisi (logits, labels)
        
    Returns:
        dict: Dictionary berisi metrik 'accuracy' dan 'f1_macro'
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average='macro', zero_division=0),
    }

# 3 combinations with freeze & LR variations
# Hyperparameter combinations for RoBERTa fine-tuning with different approaches:
# 1. Freeze 4 lower layers, LR 2e-5, batch size 16
# 2. Freeze 6 lower layers, LR 3e-5, batch size 16
# 3. Freeze 4 lower layers, LR 1.5e-5, batch size 32 (larger batch size)
# RoBERTa combos intentionally few as each fine-tuning expensive; variations focus on LR, batch, warmup, freeze.
param_combos_C = [
    {'learning_rate': 2e-5,  'batch_size': 16, 'warmup_ratio': 0.10,
     'epochs': 5, 'freeze_layers': 4},                   # 5 epoch, freeze 4
    {'learning_rate': 3e-5,  'batch_size': 16, 'warmup_ratio': 0.06,
     'epochs': 4, 'freeze_layers': 6},                   # alternatif
    {'learning_rate': 1.5e-5,'batch_size': 32, 'warmup_ratio': 0.10,
     'epochs': 5, 'freeze_layers': 4},                   # bs besar, LR kecil
]

best_val_f1_C, best_trainer_C, best_params_C = -1, None, None
t_start_C = time.time()

# Load base model ONCE
# Base model loaded once then deepcopied per combo to save download/loading time.
base_model_C = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_C, num_labels=3, ignore_mismatched_sizes=True
)

# Each combo starts from same pretrained weights for fair hyperparameter comparison.
for combo in param_combos_C:
    lr   = combo['learning_rate']
    bs   = combo['batch_size']
    wr   = combo['warmup_ratio']
    ep   = combo['epochs']
    frz  = combo['freeze_layers']

    print(f"\n lr={lr}, bs={bs}, warmup={wr}, epochs={ep}, freeze={frz}")
    t_combo = time.time()

    # Build model with mean-pooling head
    # Deepcopy prevents fine-tuning one combo polluting next combo's initial weights.
    base_copy = copy.deepcopy(base_model_C)
    model_c   = RoBERTaMeanPoolHead(base_copy, num_labels=3, dropout_rate=0.2)

    # Dynamic freeze: freeze layers 0 .. (frz-1)
    # freeze_layers=4 → train layers 4-11 + head
    # freeze_layers=6 → train layers 6-11 + head
    # Freeze lower layers reduce VRAM/overfitting risk while still training upper task-specific layers.
    for name, param in model_c.named_parameters():
        if 'classifier' in name:
            param.requires_grad = True                   # head selalu trainable
            continue
        if 'pooler' in name:
            param.requires_grad = True
            continue
        if 'embeddings' in name:
            param.requires_grad = False                  # embeddings selalu freeze
            continue
        if 'encoder.layer.' in name:
            layer_num = int(name.split('encoder.layer.')[1].split('.')[0])
            if layer_num < frz:
                param.requires_grad = False
            else:
                param.requires_grad = True

    trainable = sum(p.numel() for p in model_c.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model_c.parameters())
    print(f"    Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M "
          f"({100*trainable/total:.1f}%)")

    # Training Arguments
    # Gradient accumulation keeps effective batch large despite per-device batch limited by VRAM.
    grad_accum = 2 if bs == 16 else 1                    # effective bs ≈ 32
    steps_per_epoch = math.ceil(len(train_dataset_C) / (bs * grad_accum))
    total_steps     = steps_per_epoch * ep
    warmup_steps    = max(1, int(total_steps * wr))

    # TrainingArguments configures evaluation strategy, checkpointing, scheduler, mixed precision optimization.
    args_c = TrainingArguments(
        output_dir=f'./results_C_lr{lr}_bs{bs}_frz{frz}',
        num_train_epochs=ep,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=128,
        # fp16 leverages T4 Tensor Cores to speed training and save memory.
        fp16=True,
        learning_rate=lr,
        weight_decay=0.03,
        # Warmup prevents large updates early in fine-tuning when head still unstable.
        warmup_steps=warmup_steps,
        max_grad_norm=0.5,
        gradient_accumulation_steps=grad_accum,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",                           # enable for early stop
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,
        load_best_model_at_end=True,                     # load best checkpoint
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        disable_tqdm=False,
        seed=42,
    )

    # Early stopping: stop if 2 consecutive epochs don't improve
    early_stop_cb = EarlyStoppingCallback(early_stopping_patience=2)

    # Trainer menerima class_weights dan smoothing agar loss custom aktif selama train/eval.
    trainer_c = WeightedLabelSmoothingTrainer(
        model=model_c,
        args=args_c,
        train_dataset=train_dataset_C,
        eval_dataset=val_dataset_C,
        compute_metrics=compute_metrics,
        class_weights=class_weights_C,
        smoothing=0.05,
        base_lr=lr,                                      # untuk LLRD
        callbacks=[early_stop_cb],
    )
    trainer_c.train()

    val_metrics = trainer_c.evaluate(eval_dataset=val_dataset_C)
    elapsed = time.time() - t_combo
    print(f"    Val F1: {val_metrics['eval_f1_macro']:.4f} | "
          f"Val Acc: {val_metrics['eval_accuracy']:.4f} | "
          f"Time: {elapsed:.1f}s")

    # Model terbaik dipilih berdasarkan validation macro F1 karena targetnya seimbang antar kelas.
    if val_metrics['eval_f1_macro'] > best_val_f1_C:
        best_val_f1_C  = val_metrics['eval_f1_macro']
        best_trainer_C = trainer_c
        best_params_C  = combo

    # Clear VRAM between combos
    # Large objects deleted between combos so next checkpoint doesn't run out of VRAM/RAM.
    del model_c, trainer_c, base_copy
    torch.cuda.empty_cache()
    gc.collect()

print_best("Model C: RoBERTa", best_params_C, "Val F1", best_val_f1_C,
           time.time()-t_start_C)

# Final evaluation
# Final evaluation memakai trainer terbaik yang sudah memuat checkpoint terbaik dari validation.
test_pred_C    = best_trainer_C.predict(test_dataset_C)
y_pred_test_C  = np.argmax(test_pred_C.predictions, axis=-1)
y_pred_train_C = np.argmax(
    best_trainer_C.predict(train_dataset_C).predictions, axis=-1
)

res_C = log_result(
    "Model C: RoBERTa",
    f"{int((1-SPLIT_TEST_RATIO_C)*100)}:{int(SPLIT_TEST_RATIO_C*100)}",
    "RoBERTa+MeanPool+LLRD+LabelSmooth+EarlyStop",
    y_tr_C.values, y_pred_train_C,
    y_te_C.values, y_pred_test_C,
    best_params_C,
)
print(f"  Test Acc: {res_C['Test Accuracy']:.4f} | F1: {res_C['F1-Score']:.4f}")
plot_confusion(y_te_C.values, y_pred_test_C,
               "Confusion Matrix — Model C (RoBERTa)")

In [9]:
# CELL 9: COMPARISON TABLE + VISUALIZATION + INFERENCE RESULTS + ENSEMBLE VOTING

torch.cuda.empty_cache()

# Initial table shows individual model performance before ensemble results added.
df_comparison = pd.DataFrame(experiment_results).sort_values("Test Accuracy", ascending=False)
print("\nPERBANDINGAN MODEL")
print(df_comparison.to_string(index=False))

# Ensemble used to reduce single model weakness by combining three architecture predictions.
# ENSEMBLE: Majority Voting from 3 models
# Use test set from Model C (split 75:25) as reference
print("\n" + "="*60)
print("ENSEMBLE: Majority Voting (3 Model)")
print("="*60)

# Since splits differ, use Model C test set
# Re-predict all test sets with all models
# For fairness, use same test set (from Model C)
# Model C test set used as common reference so voting calculated on same samples.
y_test_ensemble = y_te_C.values

# Predict Model A on test set C
# Model A and B re-predicted on X_te_C so all voting candidates aligned.
y_pred_A_on_C = predict_dl_model(best_model_A, TextCNNDataset, X_te_C, vocab=vocab_A, max_len=MAX_LEN_A)
y_pred_B_on_C = predict_dl_model(best_model_B, BiLSTMDataset, X_te_C, vocab=vocab_B, max_len=MAX_LEN_B)
y_pred_C_on_C = y_pred_test_C

# Majority Voting
# scipy.stats.mode takes label chosen most per datum; with 3 models ties rare.
from scipy import stats
stacked = np.stack([y_pred_A_on_C, y_pred_B_on_C, y_pred_C_on_C], axis=0)
y_pred_ensemble, _ = stats.mode(stacked, axis=0)
y_pred_ensemble = y_pred_ensemble.flatten()

# Ensemble metrics calculated with macro average so each class contribution visible.
ens_acc = accuracy_score(y_test_ensemble, y_pred_ensemble)
ens_f1  = f1_score(y_test_ensemble, y_pred_ensemble, average='macro', zero_division=0)
ens_prec = precision_score(y_test_ensemble, y_pred_ensemble, average='macro', zero_division=0)
ens_rec  = recall_score(y_test_ensemble, y_pred_ensemble, average='macro', zero_division=0)

print(f"  Ensemble Accuracy : {ens_acc:.4f}")
print(f"  Ensemble Precision: {ens_prec:.4f}")
print(f"  Ensemble Recall   : {ens_rec:.4f}")
print(f"  Ensemble F1-Score : {ens_f1:.4f}")

# Tambahkan ke tabel
# Ensemble result uses same dict format as log_result so can be merged to final table.
ensemble_result = {
    "Model": "ENSEMBLE (Voting)", "Split": "75:25",
    "Fitur": "Majority Voting 3 Model",
    "Best Params": "TextCNN + BiLSTM+Attn + RoBERTa",
    "Train Accuracy": "-", "Test Accuracy": ens_acc,
    "Precision": ens_prec, "Recall": ens_rec, "F1-Score": ens_f1,
}
experiment_results.append(ensemble_result)

df_final = pd.DataFrame(experiment_results).sort_values("Test Accuracy", ascending=False)
print("\nHASIL AKHIR (termasuk ENSEMBLE)")
print(df_final.to_string(index=False))

# Comparison plot
# Bar plot enables quick comparison between accuracy and macro F1 across models.
fig, ax = plt.subplots(figsize=(10, 6))
df_final.set_index("Model")[["Test Accuracy", "F1-Score"]].plot(
    kind='bar', ax=ax, color=['#4C72B0', '#DD8452'], rot=20
)
ax.set_title("Perbandingan Model")
ax.set_ylabel("Score"); ax.set_ylim(0.5, 1.0)
plt.tight_layout(); plt.show()

plot_confusion(y_test_ensemble, y_pred_ensemble, "Confusion Matrix — ENSEMBLE")

# WordCloud
# WordCloud helps qualitative inspection of dominant words for weak labeling results.
for label_value, cmap in [('positive','Greens'), ('neutral','Greys'), ('negative','Reds')]:
    """Visualisasi WordCloud untuk setiap kelas sentimen.
    
    WordCloud shows words most frequently appearing for each sentiment class:
    - Positive: green color
    - Neutral: gray color
    - Negative: red color
    
    Args:
        label_value (str): Kelas sentimen ('positive', 'neutral', 'negative')
        cmap (str): Colormap untuk visualisasi
    """
    text_subset = ' '.join(df_raw[df_raw['polarity'] == label_value]['text_clean'])
    wc = WordCloud(width=800, height=400, background_color='white', colormap=cmap).generate(text_subset)
    plt.figure(figsize=(8, 4))
    plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
    plt.title(f"WordCloud — '{label_value}'"); plt.show()

# Inference
# Inference example function runs same pipeline: clean → ids → logits → probabilities.
def predict_textcnn(text_review, model, vocab, max_len):
    """Melakukan prediksi sentimen pada teks ulasan menggunakan model TextCNN.
    
    This function cleans text, converts to index sequence, and performs prediction
    using TextCNN model. Returns sentiment label and confidence score.
    
    Args:
        text_review (str): Teks ulasan yang akan diprediksi
        model (nn.Module): Model TextCNN yang sudah dilatih
        vocab (dict): Vocabulary untuk konversi teks ke indeks
        max_len (int): Panjang maksimum urutan
        
    Returns:
        tuple: (label, confidence) - label sentimen dan confidence score
    """
    # Inference cleaning must match training for consistent input distribution.
    cleaned = clean_text_light(text_review)
    ids = text_to_ids(cleaned, vocab, max_len)
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    model.eval()
    # no_grad and autocast speed inference since no backward pass.
    with torch.no_grad(), torch.amp.autocast('cuda'):
        logits = model(x)
        probs = torch.softmax(logits.float(), dim=-1).squeeze().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return ID2LABEL[pred_idx], float(probs[pred_idx])

# Example reviews include positive, negative, neutral for manual sanity check.
test_reviews = [
    "The storyline is super amazing and the English voice acting is top-tier!",
    "Too many bugs after the update, the game keeps crashing on the loading screen.",
    "The game is okay, average gacha mechanics."
]

print("\n=== HASIL INFERENCE (Model A: TextCNN) ===")
for review in test_reviews:
    sentimen, conf = predict_textcnn(review, best_model_A, vocab_A, MAX_LEN_A)
    print(f'Ulasan   : "{review}"')
    print(f"Sentimen : {sentimen.upper()} (Confidence: {conf*100:.2f}%)")
    print("-" * 50)

# Final memory summary helps detect remaining GPU allocation after all evaluation done.
if torch.cuda.is_available():
    print(f"\nGPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")